# YOLO One-Class Cow Detector Training

Convert VIA annotations into a YOLO dataset, split by video ID,
train a one-class detector, and run a small validation sample.



In [1]:
import ast
import json
import shutil
from collections import defaultdict
from pathlib import Path
from random import Random
from typing import Any

import cv2
import pandas as pd
import torch
import yaml
from ultralytics import YOLO




Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/robin/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# Configuration
CSV_PATH = Path("data/CBVD-5.csv")
IMG_ROOT = Path("data/labelframes/labelframes")
OUT_ROOT = Path("workdir/yolo_cow_oneclass")
YOLO_RUNS_DIR = Path("artifacts/runs")

EPOCHS = 30
IMG_SIZE = 640
MODEL_NAME = "yolo11n.pt"
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.2
TEST_SPLIT = 0.1
SEED = 42

if abs((TRAIN_SPLIT + VAL_SPLIT + TEST_SPLIT) - 1.0) > 1e-9:
    raise ValueError("Train/val/test splits must sum to 1.0")

torch.manual_seed(SEED)




In [3]:
def parse_file_list(file_list_field: Any) -> list[str]:
    """Parse VIA file_list field into a list of file names."""
    if isinstance(file_list_field, list):
        return [str(x) for x in file_list_field]

    if file_list_field is None:
        return []

    try:
        parsed = ast.literal_eval(str(file_list_field))
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except (ValueError, SyntaxError):
        pass

    text = str(file_list_field).strip()
    return [text] if text else []


def parse_box(spatial_coordinates: Any) -> tuple[float, float, float, float]:
    """Extract VIA xywh box from spatial coordinates."""
    coords = (
        json.loads(spatial_coordinates)
        if isinstance(spatial_coordinates, str)
        else spatial_coordinates
    )
    if not isinstance(coords, list) or not coords:
        raise ValueError("Invalid spatial_coordinates format")

    if isinstance(coords[0], list):
        coords = coords[0]
    if len(coords) < 5:
        raise ValueError("Expected [shape, x, y, w, h]")

    _, x, y, w, h = coords
    return float(x), float(y), float(w), float(h)


def video_id_from_name(image_name: str) -> str:
    """Extract video identifier from frame name (e.g., 618_00002.jpg -> 618)."""
    stem = Path(image_name).stem
    return stem.split("_")[0] if "_" in stem else stem


def to_yolo_norm(
    x: float,
    y: float,
    w: float,
    h: float,
    width: int,
    height: int,
) -> tuple[float, float, float, float]:
    """Convert absolute xywh to YOLO normalized cxcywh."""
    cx = (x + w / 2.0) / width
    cy = (y + h / 2.0) / height
    return cx, cy, w / width, h / height


def resolve_device() -> str:
    if torch.cuda.is_available():
        print(f"Training on CUDA: {torch.cuda.get_device_name(0)}")
        return "cuda"
    if torch.backends.mps.is_available():
        print("Training on Apple Silicon GPU (MPS)")
        return "mps"
    print("Training on CPU")
    return "cpu"




In [4]:
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")
if not IMG_ROOT.exists():
    raise FileNotFoundError(f"Image root not found: {IMG_ROOT}")

print(f"Dataset: {CSV_PATH} -> {OUT_ROOT}")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
YOLO_RUNS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH, skiprows=9)
required_columns = {"file_list", "spatial_coordinates"}
if not required_columns.issubset(df.columns):
    raise ValueError(
        f"CSV missing required columns: {required_columns - set(df.columns)}"
    )

boxes_by_image: dict[str, list[tuple[float, float, float, float]]] = defaultdict(list)
skipped_rows = 0

for _, row in df.iterrows():
    try:
        files = parse_file_list(row["file_list"])
        if not files:
            skipped_rows += 1
            continue

        image_name = files[0]
        x, y, w, h = parse_box(row["spatial_coordinates"])
        boxes_by_image[image_name].append((x, y, w, h))
    except Exception:
        skipped_rows += 1




Dataset: data/CBVD-5.csv -> workdir/yolo_cow_oneclass


In [5]:
rng = Random(SEED)
video_ids = sorted({video_id_from_name(name) for name in boxes_by_image.keys()})
rng.shuffle(video_ids)

num_videos = len(video_ids)
num_train = int(num_videos * TRAIN_SPLIT)
num_val = int(num_videos * VAL_SPLIT)

video_splits = {
    "train": set(video_ids[:num_train]),
    "val": set(video_ids[num_train : num_train + num_val]),
    "test": set(video_ids[num_train + num_val :]),
}


def get_split(image_name: str) -> str:
    vid = video_id_from_name(image_name)
    for split, id_set in video_splits.items():
        if vid in id_set:
            return split
    return "test"


print(
    f"Videos split -> train: {len(video_splits['train'])}, "
    f"val: {len(video_splits['val'])}, test: {len(video_splits['test'])}"
)
print(f"Annotated images: {len(boxes_by_image)} | skipped rows: {skipped_rows}")




Videos split -> train: 375, val: 107, test: 55
Annotated images: 3199 | skipped rows: 0


In [6]:
for split_name in ["train", "val", "test"]:
    (OUT_ROOT / "images" / split_name).mkdir(parents=True, exist_ok=True)
    (OUT_ROOT / "labels" / split_name).mkdir(parents=True, exist_ok=True)

copied_images = 0
written_labels = 0
missing_images = 0

for image_name, boxes in boxes_by_image.items():
    src_path = IMG_ROOT / image_name
    if not src_path.exists():
        missing_images += 1
        continue

    image = cv2.imread(str(src_path))
    if image is None:
        missing_images += 1
        continue

    height, width = image.shape[:2]
    split_name = get_split(image_name)

    dst_img_path = OUT_ROOT / "images" / split_name / image_name
    shutil.copy2(src_path, dst_img_path)
    copied_images += 1

    yolo_lines = []
    for x, y, w, h in boxes:
        cx, cy, nw, nh = to_yolo_norm(x, y, w, h, width, height)
        cx = max(0.0, min(1.0, cx))
        cy = max(0.0, min(1.0, cy))
        nw = max(1e-6, min(1.0, nw))
        nh = max(1e-6, min(1.0, nh))
        yolo_lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

    dst_label_path = OUT_ROOT / "labels" / split_name / f"{Path(image_name).stem}.txt"
    dst_label_path.write_text("\n".join(yolo_lines), encoding="utf-8")
    written_labels += 1

data_yaml = {
    "path": str(OUT_ROOT.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "cow"},
    "nc": 1,
}
with (OUT_ROOT / "data.yaml").open("w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(
    f"Prepared YOLO dataset -> copied images: {copied_images}, "
    f"labels: {written_labels}, missing images: {missing_images}"
)




Prepared YOLO dataset -> copied images: 3199, labels: 3199, missing images: 0


In [7]:
device = resolve_device()
model = YOLO(MODEL_NAME)
_ = model.train(
    data=str(OUT_ROOT / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    device=device,
    project=str(YOLO_RUNS_DIR),
    name="yolo_oneclass",
    exist_ok=True,
    verbose=False,
)
print(
    "Training complete. Model artifact expected in "
    f"{YOLO_RUNS_DIR}/detect/yolo_oneclass/weights/best.pt"
)




Training on CUDA: NVIDIA GeForce RTX 4080
Ultralytics 8.4.14 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4080, 16376MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=workdir/yolo_cow_oneclass/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_oneclass, nbs=64, nms=False, opset=None, optimize=

In [9]:
try:
    candidate_models = sorted(YOLO_RUNS_DIR.rglob("weights/best.pt"))
    if not candidate_models:
        print(f"No trained best.pt found under {YOLO_RUNS_DIR}")
    else:
        best_model_path = candidate_models[-1]
        detector = YOLO(str(best_model_path))

        val_images = list((OUT_ROOT / "images" / "val").glob("*.jpg"))[:3]
        total_detections = 0
        for img_path in val_images:
            pred = detector.predict(source=str(img_path), conf=0.25, verbose=False)[0]
            detections = len(pred.boxes) if pred.boxes is not None else 0
            total_detections += detections
            print(f"{img_path.name}: {detections} cows")

        if val_images:
            print(
                f"Validation sample: {total_detections} detections across "
                f"{len(val_images)} images"
            )
        print(f"Latest model: {best_model_path}")
except Exception as e:
    print(f"Validation sample failed: {e}")


249_00002.jpg: 15 cows
104_00002.jpg: 14 cows
533_00007.jpg: 6 cows
Validation sample: 35 detections across 3 images
Latest model: artifacts/runs/detect/yolo_oneclass/weights/best.pt
